In [1]:
# Instalar biblioteca pymongo en Colab.
!pip install pymongo

# Conexión con Secret
from google.colab import userdata
from pymongo import MongoClient

uri = userdata.get("MONGO_URI")
client = MongoClient(uri)
client.admin.command("ping")
print("Conexión creada")

db = client["ecommify_test"]
productos = db["productos"]
reviews = db["reviews"]
geolocation = db["geolocation"]

print("Productos:", productos.count_documents({}))
print("Reviews:", reviews.count_documents({}))
print("Geolocation:", geolocation.count_documents({}))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 24.2 MB/s eta 0:00:00
Conexión creada
Productos: 1201
Reviews: 5000
Geolocation: 19015


In [38]:
import json

########################### Consultar review de producto
reviews_by_product = db.command(
    'explain',
    {
        'find': 'reviews',
        'filter': {
            'product_id': '93c480c7d11c68ba0a71e850da61b674',
        },
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(reviews_by_product['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)

########################### Consultar reviews de productocon score = 5
reviews_by_product_score = db.command(
    'explain',
    {
        'find': 'reviews',
        'filter': {
            'product_id': '93c480c7d11c68ba0a71e850da61b674',
            'score': 5,
        },
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(reviews_by_product_score['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)


######################## Consultar productos por categoria
products_by_category = db.command(
    'explain',
    {
        'find': 'productos',
        'filter': {
            'category': 'beleza_saude',
        },
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(products_by_category['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)


######################## Consultar productos por categoría y ordena/agrupa por rating.
products_by_category_rating = db.command(
    'explain',
    {
        'find': 'productos',
        'filter': {
            'category': 'cama_mesa_banho',
            'computed_metrics.average_rating': {'$gte': 4},
        },
        'sort': {'computed_metrics.total_units_sold': -1},
        'limit': 20,
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(products_by_category_rating['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)



######################## Consultar productos más vendidos con rating mayor o igual a 4
products_by_category_rating_4 = db.command(
    'explain',
    {
        'find': 'productos',
        'filter': {
            'computed_metrics.average_rating': {'$gte': 4},
        },
        'sort': {'computed_metrics.total_units_sold': -1},
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(products_by_category_rating_4['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)

######################### Busqueda de productos por texto

## sin indice
products_search = db.command(
    'explain',
    {
        'find': 'productos',
        'filter': {
            'name':'Producto Ecommify 08574b07',
        },
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(products_search['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)

## con indice
products_search = db.command(
    'explain',
    {
        'find': 'productos',
        'filter': {
            '$text': {
                '$search': '08574b07',
            },
        },
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(products_search['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)

######################### ubicaciones cercanas

## Sin indice

# Aproximación de un rango de 10km usando límites de latitud y longitud
# 1 grado de latitud ~ 111 km
# 1 grado de longitud en -23.5333 grados de latitud ~ 101.6 km (111 * cos(-23.5333))
# Por lo tanto, 10km es aproximadamente 0.09 grados de latitud y 0.098 grados de longitud

center_lon = -46.6333
center_lat = -23.5333

degree_lat_approx_10km = 10 / 111.0
degree_lon_approx_10km = 10 / (111.0 * abs(3.141592653589793 * (center_lat / 180.0)))
degree_lon_approx_10km = 10 / (111.0 * abs(0.9169))

min_lat = center_lat - degree_lat_approx_10km
max_lat = center_lat + degree_lat_approx_10km
min_lon = center_lon - degree_lon_approx_10km
max_lon = center_lon + degree_lon_approx_10km

geolocation_bbox_no_index = db.command(
    'explain',
    {
        'find': 'geolocation',
        'filter': {
            'location.coordinates.0': {'$gte': min_lon, '$lte': max_lon}, # Longitude
            'location.coordinates.1': {'$gte': min_lat, '$lte': max_lat},  # Latitude
        },
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado_bbox = json.dumps(geolocation_bbox_no_index['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('Execution stats for bounding box query without 2dsphere index:')
print(json_formateado_bbox)

## Usando indice

geolocation_near = db.command(
    'explain',
    {
        'find': 'geolocation',
        'filter': {
            'location': {
                '$near': {
                    '$geometry': {
                        'type': 'Point',
                        'coordinates': [-46.6333, -23.5333],
                    },
                    '$maxDistance': 10000,
                    '$minDistance': 0,
                },
            },
        },
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(geolocation_near['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)

######################### Búsqueda dentro de un área

geolocation_area = db.command(
    'explain',
    {
        'find': 'geolocation',
        'filter': {
            'location': {
                '$geoWithin': {
                    '$geometry': {
                        'type': 'Polygon',
                        'coordinates': [
                               [-46.6333, -23.5333],
                               [-46.6333, -23.5333],
                               [-46.6333, -23.5333],
                               [-46.6333, -23.5333],
                               [-46.6333, -23.5333],
                        ],
                    },
                },
            },
        }
    },
    verbosity='executionStats',
)

# Convertir a string formateado con indentación de 4 espacios
json_formateado = json.dumps(geolocation_area['executionStats']['executionStages'], indent=4, ensure_ascii=False)
print('executionTimeMillis:',json_formateado)


executionTimeMillis: {
    "isCached": false,
    "stage": "FETCH",
    "nReturned": 116,
    "executionTimeMillisEstimate": 0,
    "works": 117,
    "advanced": 116,
    "needTime": 0,
    "needYield": 0,
    "saveState": 0,
    "restoreState": 0,
    "isEOF": 1,
    "docsExamined": 116,
    "alreadyHasObj": 0,
    "inputStage": {
        "stage": "IXSCAN",
        "nReturned": 116,
        "executionTimeMillisEstimate": 0,
        "works": 117,
        "advanced": 116,
        "needTime": 0,
        "needYield": 0,
        "saveState": 0,
        "restoreState": 0,
        "isEOF": 1,
        "keyPattern": {
            "category": 1
        },
        "indexName": "category_1",
        "isMultiKey": false,
        "multiKeyPaths": {
            "category": []
        },
        "isUnique": false,
        "isSparse": false,
        "isPartial": false,
        "indexVersion": 2,
        "direction": "forward",
        "indexBounds": {
            "category": [
                "[\"bele

In [37]:

# Crear índices

# Índice simple sobre product_id en reviews
reviews.create_index([
    ('product_id', 1),
])

# Índice compuesto sobre product_id y score en reviews
reviews.create_index([
    ('product_id', 1),
    ('score', -1),
])

# Índice simple sobre category en productos
productos.create_index([
    ('category', 1),
])

# Índice compuesto por category y computed_metrics.average_rating en productos
productos.create_index([
    ('category', 1),
    ('computed_metrics.average_rating', -1),
])

# Índice compuesto por category y price en productos
productos.create_index([
    ('category', 1),
    ('price', 1),
])

# Índice parcial en productos
productos.create_index(
    [('computed_metrics.total_units_sold', -1)],
    partialFilterExpression={'computed_metrics.average_rating': {'$gte': 4}},
    name='idx_top_sellers_rated',
)

# Índice de texto para busqueda de productos
productos.create_index([
    ('name', 'text'),
    ('category', 'text'),
])

# Índice de Geo para consultas geoespaciales
geolocation.create_index([
     ('location', '2dsphere'),
])



'location_2dsphere'

In [32]:
import time

# Agregación: promedio de rating por categoría (top 5)
pipeline = [
    {"$match": {"computed_metrics.average_rating": {"$ne": None}}},
    {"$group": {"_id": "$category", "promedio_rating": {"$avg": "$computed_metrics.average_rating"}}},
    {"$sort": {"promedio_rating": -1}},
    {"$limit": 5},
]
for row in productos.aggregate(pipeline):
    print(row)

#Pipeline optimizado con 5+ stages ($match, $lookup, $unwind, $group, $addFields, $sort, $limit)

t0 = time.perf_counter()

pipeline_u5 = [
    {'$match': {'computed_metrics.average_rating': {'$gte': 4}}},
    {'$lookup': {
        'from': 'reviews',
        'localField': 'product_id',
        'foreignField': 'product_id',
        'as': 'product_reviews',
    }},
    {'$unwind': '$product_reviews'},
    {'$group': {
        '_id': '$category',
        'promedio_rating': {'$avg': '$computed_metrics.average_rating'},
        'total_reviews': {'$sum': 1},
        'ventas_promedio': {'$avg': '$computed_metrics.total_units_sold'},
    }},
    {'$addFields': {
        'score_compuesto': {'$multiply': ['$promedio_rating', '$ventas_promedio']},
    }},
    {'$sort': {'score_compuesto': -1}},
    {'$limit': 10},
]

result = list(productos.aggregate(pipeline_u5, allowDiskUse=True))

print(f'Tiempo pipeline: {(time.perf_counter() - t0) * 1000:.2f} ms')
for row in result[:5]:
    print(row)

{'_id': 'cds_dvds_musicais', 'promedio_rating': 4.67}
{'_id': 'climatizacao', 'promedio_rating': 4.47}
{'_id': 'eletrodomesticos', 'promedio_rating': 4.4325}
{'_id': 'portateis_casa_forno_e_cafe', 'promedio_rating': 4.37}
{'_id': 'livros_interesse_geral', 'promedio_rating': 4.36}
Tiempo pipeline: 2229.41 ms
{'_id': 'ferramentas_jardim', 'promedio_rating': 4.1721097046413504, 'total_reviews': 237, 'ventas_promedio': 269.9957805907173, 'score_compuesto': 1126.4520164147484}
{'_id': 'moveis_decoracao', 'promedio_rating': 4.185444444444444, 'total_reviews': 180, 'ventas_promedio': 205.72222222222223, 'score_compuesto': 861.0389320987655}
{'_id': 'relogios_presentes', 'promedio_rating': 4.1984976525821605, 'total_reviews': 213, 'ventas_promedio': 125.2018779342723, 'score_compuesto': 525.6597906059204}
{'_id': 'informatica_acessorios', 'promedio_rating': 4.281592592592593, 'total_reviews': 270, 'ventas_promedio': 118.91851851851852, 'score_compuesto': 509.16064801097394}
{'_id': 'beleza_sau